In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler , OneHotEncoder
from sklearn.compose import ColumnTransformer , make_column_selector , TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error ,root_mean_squared_error
%matplotlib inline

In [29]:
df = pd.read_csv('automobiles.csv')

X = df.drop(columns=['price'])
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [16]:
df.columns

Index(['symboling', 'normalized-losses', 'make', 'fuel-type', 'aspiration',
       'num-of-doors', 'body-style', 'drive-wheels', 'engine-location',
       'wheel-base', 'length', 'width', 'height', 'curb-weight', 'engine-type',
       'num-of-cylinders', 'engine-size', 'fuel-system', 'bore', 'stroke',
       'compression-ratio', 'horsepower', 'peak-rpm', 'city-mpg',
       'highway-mpg', 'price'],
      dtype='object')

In [6]:
def engineer_features(dataframe):
    df_out = dataframe.copy()
    
    df_out['combined_mpg'] = 2 / ((1 / df_out['city-mpg']) + (1 / df_out['highway-mpg']))
    
    df_out['power_index'] = df_out['horsepower'] / df_out['city-mpg']
    
    return df_out

X_train_engineered = engineer_features(X_train)
X_test_engineered = engineer_features(X_test)

In [7]:
numeric_features = ['horsepower', 'combined_mpg', 'power_index']
categorical_features = ['make', 'fuel-type', 'body-style']

num_pipeline = make_pipeline(
    SimpleImputer(strategy='median'), 
    StandardScaler()                  
)

cat_pipeline = make_pipeline(
    SimpleImputer(strategy='most_frequent'), 
    OneHotEncoder(handle_unknown='ignore', sparse_output=False) 
)

preprocessing = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, numeric_features),
        ('cat', cat_pipeline, categorical_features)
    ]
)

In [9]:
base_pipeline = make_pipeline(preprocessing, Ridge(alpha=5.0))

final_model = TransformedTargetRegressor(
    regressor=base_pipeline,
    func=np.log1p,       
    inverse_func=np.expm1 
)

final_model.fit(X_train_engineered, y_train)

TransformedTargetRegressor(func=<ufunc 'log1p'>, inverse_func=<ufunc 'expm1'>,
                           regressor=Pipeline(steps=[('columntransformer',
                                                      ColumnTransformer(transformers=[('num',
                                                                                       Pipeline(steps=[('simpleimputer',
                                                                                                        SimpleImputer(strategy='median')),
                                                                                                       ('standardscaler',
                                                                                                        StandardScaler())]),
                                                                                       ['horsepower',
                                                                                        'combined_mpg',
                                                                                        'power_index']),
                                                                                      ('cat',
                                                                                       Pipeline(steps=[('simpleimputer',
                                                                                                        SimpleImputer(strategy='most_frequent')),
                                                                                                       ('onehotencoder',
                                                                                                        OneHotEncoder(handle_unknown='ignore',
                                                                                                                      sparse_output=False))]),
                                                                                       ['make',
                                                                                        'fuel-type',
                                                                                        'body-style'])])),
                                                     ('ridge',
                                                      Ridge(alpha=5.0))]))

In [12]:
predictions = final_model.predict(X_test_engineered)

mae = mean_absolute_error(y_test, predictions)
rmse = root_mean_squared_error(y_test, predictions)

print(f"Operational Baseline Established.")
print(f"Final Test MAE: ${mae:.2f}")
print(f"Final Test RMSE: ${rmse:.2f}\n")

for i in range(5):
    print(f"Car {i+1} -> Predicted: ${predictions[i]:.2f} | Actual: ${y_test.values[i]:.2f}")

Operational Baseline Established.
Final Test MAE: $1693.31
Final Test RMSE: $2444.96

Car 1 -> Predicted: $10953.50 | Actual: $11549.00
Car 2 -> Predicted: $6624.32 | Actual: $6338.00
Car 3 -> Predicted: $12410.60 | Actual: $8449.00
Car 4 -> Predicted: $6181.16 | Actual: $6189.00
Car 5 -> Predicted: $15635.56 | Actual: $22018.00


In [13]:
!pip install gradio

  Using cached brotli-1.2.0-cp312-cp312-win_amd64.whl.metadata (6.3 kB)
  Using cached groovy-0.1.2-py3-none-any.whl.metadata (6.1 kB)
  Using cached pydub-0.25.1-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached safehttpx-0.1.7-py3-none-any.whl.metadata (4.2 kB)
  Using cached semantic_version-2.10.0-py2.py3-none-any.whl.metadata (9.7 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
   ---------------------------------------- 0.0/31.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/31.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/31.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/31.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/31.3 MB ? eta -:--:--
   ---------------------------------------- 0.3/31.3 MB ? eta -:--:--
   ---------------------------------------- 0.3/31.3 MB ? eta -:--:--
    -----------------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.37.1 requires protobuf<6,>=3.20, but you have protobuf 6.33.2 which is incompatible.
streamlit 1.37.1 requires rich<14,>=10.14.0, but you have rich 15.0.0 which is incompatible.


In [30]:
import gradio as gr
import pandas as pd

def predict_car_price(make, doors, horsepower, city_mpg, highway_mpg, fuel_type, body_style):
    input_data = pd.DataFrame([{
        'make': make.lower(),
        'num-of-doors': doors.lower(), 
        'horsepower': float(horsepower),    
        'city-mpg': float(city_mpg),
        'highway-mpg': float(highway_mpg),
        'fuel-type': fuel_type.lower(),
        'body-style': body_style.lower()
    }])
    
    input_data['combined_mpg'] = 2 / ((1 / input_data['city-mpg']) + (1 / input_data['highway-mpg']))
    input_data['power_index'] = input_data['horsepower'] / input_data['city-mpg']
    
    prediction = final_model.predict(input_data)
    
    return f"${prediction[0]:,.2f}"

available_makes = sorted(df['make'].dropna().unique())
available_fuels = sorted(df['fuel-type'].dropna().unique())
num_of_doors = sorted(df['num-of-doors'].dropna().unique())
available_styles = sorted(df['body-style'].dropna().unique())

interface = gr.Interface(
    fn=predict_car_price,
    inputs=[
        gr.Dropdown(choices=available_makes, label="Car Brand / Make", value="toyota"), 
        gr.Dropdown(choices=num_of_doors, label='Number of Doors', value=num_of_doors[0]), 
        gr.Slider(minimum=40, maximum=300, step=1, value=100, label="Horsepower"),      
        gr.Slider(minimum=10, maximum=60, step=1, value=25, label="City MPG"),         
        gr.Slider(minimum=10, maximum=60, step=1, value=30, label="Highway MPG"),       
        gr.Radio(choices=available_fuels, label="Fuel Type", value="gas"),               
        gr.Dropdown(choices=available_styles, label="Body Style", value="sedan")         
    ],
    outputs=gr.Textbox(label="Estimated Market Value"),
    title="🚗 Smart Car Price Predictor Dashboard",
    description="Adjust the vehicle's mechanical specifications and brand values below to dynamically evaluate estimated prices.",
)

interface.launch(inline=True, theme='soft')

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


C:\Users\shoai\anaconda3\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
